> 在线讲义：[第 45 章 · MoE 的数学：门控与 top-k、负载均衡损失、无辅助损失的偏置、容量因子、专家并行的通信量](https://wikieden.github.io/math-and-eval-stats/lessons/45-moe-math.html) · 本书《数学与实验统计》

# 第 45 章 · MoE 的数学：门控与 top-k、负载均衡损失、无辅助损失的偏置、容量因子、专家并行的通信量

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/45-moe-math.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 门控 | top-k 归一化 | `topk_gate(logits)` | [0, 1]，[0.7311, 0.2689] |
| 第二步 辅助损失 | NΣfP | `switch_aux(f, p)` | 1.0、1.84、3.6 |
| 第三步 容量 | 二项分布 | `drop_rate(4096, 64, cf)` | 4.94%、0.13% |
| 第四步 偏置 | sign | `bias_update(load, b)` | [−0.001, 0.001, 0, 0] |
| 第五步 通信 | 2·T·k·h·B | `a2a_bytes(8192, 8, 7168)` | 1.75 GiB |

例子：一个 token 对 4 个专家的门控 logits [2, 1, 0.5, −1]；Switch 负载均衡辅助损失在均匀 / 偏斜两种分布下的值；
容量因子与丢 token 的概率（二项分布）；DeepSeek-V3 无辅助损失的偏置调节一步；专家并行 all-to-all 的通信量。
里程碑 ME-M28 的“模拟 top-2 门控的负载分布、加辅助损失 / 偏置调节后对比”留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
from scipy.stats import binom

**函数 `topk_gate`**：softmax 后取最大的 k 个，再在这 k 个里重新归一化（Mixtral 的做法）。返回 (专家编号, 权重)。

In [ ]:
def topk_gate(logits: np.ndarray, k: int = 2) -> tuple[np.ndarray, np.ndarray]:
    """softmax 后取最大的 k 个，再在这 k 个里重新归一化（Mixtral 的做法）。返回 (专家编号, 权重)。"""
    p = np.exp(logits - logits.max())
    p /= p.sum()
    idx = np.argsort(-p)[:k]
    return idx, p[idx] / p[idx].sum()

**函数 `switch_aux`**：Switch Transformer 的辅助损失 N·Σ f_i·P_i：f_i 是实际分到专家 i 的 token 比例，P_i 是路由概率的平均。均匀时 = 1。

In [ ]:
def switch_aux(f: np.ndarray, p: np.ndarray) -> float:
    """Switch Transformer 的辅助损失 N·Σ f_i·P_i：f_i 是实际分到专家 i 的 token 比例，P_i 是路由概率的平均。均匀时 = 1。"""
    return float(len(f) * (f * p).sum())

**函数 `drop_rate`**：每个专家容量 C = cf·tokens·k/E。若路由完全随机均匀，一个专家收到的 token 数 ~ Binomial(tokens·k, 1/E)，

In [ ]:
def drop_rate(tokens: int, experts: int, cf: float, k: int = 1) -> float:
    """每个专家容量 C = cf·tokens·k/E。若路由完全随机均匀，一个专家收到的 token 数 ~ Binomial(tokens·k, 1/E)，
    超过 C 的部分被丢弃。返回期望丢弃比例。"""
    n, p = tokens * k, 1 / experts
    cap = int(cf * n / experts)
    x = np.arange(cap + 1, n + 1)
    return float(((x - cap) * binom.pmf(x, n, p)).sum() / (n * p))

**函数 `bias_update`**：DeepSeek-V3：每个专家一个只用于选 top-k 的偏置 b_i；负载高于平均就减 γ，低于平均就加 γ（不进梯度、不改门控权重）。

In [ ]:
def bias_update(load: np.ndarray, b: np.ndarray, gamma: float = 0.001) -> np.ndarray:
    """DeepSeek-V3：每个专家一个只用于选 top-k 的偏置 b_i；负载高于平均就减 γ，低于平均就加 γ（不进梯度、不改门控权重）。"""
    return b - gamma * np.sign(load - load.mean())

**函数 `a2a_bytes`**：专家并行一层的 all-to-all：派发（dispatch）和汇总（combine）各发一次，每个 token 发给 k 个专家。

In [ ]:
def a2a_bytes(tokens: int, k: int, hidden: int, bytes_per: int = 2) -> int:
    """专家并行一层的 all-to-all：派发（dispatch）和汇总（combine）各发一次，每个 token 发给 k 个专家。"""
    return 2 * tokens * k * hidden * bytes_per

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 门控：softmax → top-k → 重新归一化")
idx, w = topk_gate(np.array([2.0, 1.0, 0.5, -1.0]))
print(f"  logits [2, 1, 0.5, −1]：选专家 {idx.tolist()}，权重 {np.round(w, 4).tolist()}；输出 = Σ 权重 × 专家输出")

In [ ]:
print("\n第二步 负载均衡辅助损失 N·Σ f_i P_i")
for name, f, p in (("均匀", [0.25] * 4, [0.25] * 4), ("偏斜", [0.7, 0.1, 0.1, 0.1], [0.6, 0.15, 0.15, 0.1]),
                   ("全挤到一个", [1, 0, 0, 0], [0.9, 0.05, 0.03, 0.02])):
    print(f"  {name}：f = {f}、P = {p} → {switch_aux(np.array(f), np.array(p)):.4f}")

In [ ]:
print("\n第三步 容量因子：随机路由也会溢出")
for cf in (1.0, 1.25, 2.0):
    print(f"  4096 个 token、64 个专家、top-1、容量因子 {cf}：期望丢弃 {drop_rate(4096, 64, cf):.2%}")

In [ ]:
print("\n第四步 无辅助损失的偏置调节（一步）")
load = np.array([400.0, 100.0, 250.0, 250.0])
print(f"  负载 {load.tolist()}（平均 250）：偏置 0 → {bias_update(load, np.zeros(4)).tolist()}")

In [ ]:
print("\n第五步 专家并行的通信量")
print(f"  每卡 8192 个 token、top-8、hidden 7168、bf16：一层 all-to-all {a2a_bytes(8192, 8, 7168) / 2**30:.2f} GiB")

## 3. 代码里才看得到的

1. **丢弃比例用二项分布精确算**：对超过容量的每个负载值加权求和，不用模拟。
2. **偏置不进入权重**：`bias_update` 只返回新的 b，选专家时用 p + b、算权重时仍用原来的 p。
3. **没有模拟训练中的负载变化**：top-2 门控加辅助损失 / 偏置调节后的负载分布对比留给自己做（里程碑 ME-M28）。

## 4. 自己改一改

1. 把 `drop_rate` 改成 top-2（k = 2），cf = 1 时丢弃比例变多还是变少？
2. 连续调用 `bias_update` 10 次但负载不变，偏置会怎样？这说明实际系统里负载为什么必须随偏置变化？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 门控 softmax 取 top-k，再在 k 个里归一；
> 辅助损失 N 乘 Σ f P，均匀时最小是一；
> 容量是 cf 乘平均，随机波动也会丢；
> 无辅助损失调偏置，热的减、冷的加；
> 专家并行两次 all-to-all，流量 2·tokens·k·h。